In [12]:
import pandas as pd
import json

# carregando json
with open('../dados/dados_nivel_1.json', 'r', encoding='utf-8') as f:
    dados_brutos = json.load(f)

taxa_cambio = dados_brutos['taxa_cambio_usd_brl']
df = pd.DataFrame(dados_brutos['operacoes'])

print(f"Taxa de câmbio USD/BRL: {taxa_cambio}")
print(f"Total de registros carregados: {len(df)}")
df.head()

Taxa de câmbio USD/BRL: 5.4
Total de registros carregados: 20


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,ted,transferencia_enviada,Delta Transportes,


In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   id           20 non-null     str  
 1   cliente_id   20 non-null     str  
 2   data         19 non-null     str  
 3   valor        20 non-null     int64
 4   moeda        20 non-null     str  
 5   canal        20 non-null     str  
 6   tipo         20 non-null     str  
 7   contraparte  20 non-null     str  
 8   observacao   20 non-null     str  
dtypes: int64(1), str(8)
memory usage: 1.5 KB


In [14]:
df.isnull().sum()

id             0
cliente_id     0
data           1
valor          0
moeda          0
canal          0
tipo           0
contraparte    0
observacao     0
dtype: int64

In [15]:
df.duplicated().sum()

np.int64(1)

In [16]:
print("Moedas:", df['moeda'].unique())
print("Canais:", df['canal'].unique())
print("Tipos:", df['tipo'].unique())

Moedas: <StringArray>
['BRL', 'USD']
Length: 2, dtype: str
Canais: <StringArray>
['pix', 'ted', 'boleto', 'cartao', 'especie']
Length: 5, dtype: str
Tipos: <StringArray>
['transferencia_enviada', 'pagamento', 'transferencia_recebida', 'deposito']
Length: 4, dtype: str


## Problemas encontrados:

1. Data nula (1 registro) - "OP-0017" veio sem data, com a observação "data não capturada pelo sistema". Como a Regra 1 depende de agrupar por data, esse registro não pode participar dela sem uma decisão explícita.

2. Linha duplicada (1 registro) - "OP-0017" aparece duas vezes, com todos os campos idênticos.

3. Moeda mista (BRL e USD) - a maioria dos valores está em BRL, mas "OP-0013" veio em USD. Precisa ser convertido para BRL usando a taxa fornecida antes de entrar em qualquer soma ou comparação.

In [ ]:
# Removendo a linha duplicada, mantendo a primeira ocorrência.
print(f"Linhas antes: {len(df)}")
df = df.drop_duplicates().reset_index(drop=True)
print(f"Linhas depois: {len(df)}")

Linhas antes: 20
Linhas depois: 19


In [ ]:
# Não removi nem inventei data. Crio uma flag explícita para o registro com data ausente, preservando o dado original (NaN continua NaN).
df['data_ausente'] = df['data'].isnull()

df[df['data_ausente']]

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao,data_ausente
16,OP-0017,CLI-A-5,NaN,4300,BRL,especie,deposito,Gama Distribuidora,data nao capturada pelo sistema,True


In [ ]:
# Crio a coluna valor_brl: valores já em BRL ficam como estão;
# Valores em USD são multiplicados pela taxa de câmbio fornecida no JSON.
df['valor_brl'] = df.apply(lambda row: row['valor'] * taxa_cambio if row['moeda'] == 'USD' else row['valor'], axis=1)

# Conferência: mostra apenas os registros que passaram pela conversão.
df[df['moeda'] == 'USD'][['id', 'cliente_id', 'valor', 'moeda', 'valor_brl']]

,id,cliente_id,valor,moeda,valor_brl
12,OP-0013,CLI-A-4,12000,USD,64800.0
